# Drug Shortage Forecasting — Project Walkthrough (Weeks 1–7)

**Anubhaw Goyal · June 2026**

Loads the cached pipeline outputs and reproduces the headline numbers and figures.
Run from the `notebooks/` folder of the repo with the project environment installed
(`pip install -e ".[dev]"` from the repo root). Requires the data cache
(`data/raw`, `data/interim`, `data/processed`) built by `scripts/pull_data.py` and the
pipeline modules — see `HANDOFF.md` at the project root for the full build sequence.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
INTERIM, PROCESSED, EXP = ROOT / "data/interim", ROOT / "data/processed", ROOT / "experiments"
print("repo root:", ROOT)

## 1. Entity resolution coverage (Week 2)

In [ ]:
cov = json.loads((INTERIM / "coverage.json").read_text())
print(f"shortage records: {cov['n_shortage_records']:,} | ingredients: {cov['n_shortage_ingredients']}")
print("Orange Book match:", {k: v["records"] for k, v in cov["orange_book_match"].items()})
print("NDC match:        ", {k: v["records"] for k, v in cov["ndc_match"].items()})

## 2. Reconstructed shortage spells (Week 3)

The live FDA database retains only 29 resolved records; spells are reconstructed from
44 Internet Archive snapshots (onsets exact, resolutions interval-censored).

In [ ]:
sp = pd.read_parquet(INTERIM / "shortage_spells.parquet")
print(f"ingredient spells: {len(sp):,} | resolved: {(~sp.censored).sum()} | censored: {sp.censored.sum()}")
res = sp[~sp.censored]
print(f"median resolved duration (midpoint): {res.dur_mid_days.median()/30.44:.1f} months")
sp[sp.onset >= "2018"].groupby(sp.onset.dt.year).size().plot(kind="bar", figsize=(8,3),
    title="Ingredient-level shortage onsets by year (validated vs FDA/ASHP decline)")
plt.tight_layout(); plt.show()

## 3. Survival analysis (Week 5)

In [ ]:
from lifelines import KaplanMeierFitter
pn = pd.read_parquet(PROCESSED / "panel_v2.parquet")
feats = pn[["ing_loose","month","injectable"]]
d = sp[sp.onset >= "2018-01-01"].copy()
d["onset_month"] = d.onset.values.astype("datetime64[M]")
d = d.merge(feats, left_on=["ing_loose","onset_month"], right_on=["ing_loose","month"])
end = d.end_lo + (d.end_hi - d.end_lo) / 2
d["event"] = ~d.censored
d["dur_m"] = np.where(d.event, (end - d.onset).dt.days, (pd.Timestamp("2026-06-12") - d.onset).dt.days) / 30.44
d = d[d.dur_m > 0]
fig, ax = plt.subplots(figsize=(8,5))
for label, sub in [("Injectable", d[d.injectable]), ("Non-injectable", d[~d.injectable])]:
    KaplanMeierFitter().fit(sub.dur_m, sub.event, label=f"{label} (n={len(sub)})").plot_survival_function(ax=ax)
ax.set_xlabel("Months since onset"); ax.set_ylabel("P(still in shortage)")
ax.set_title("Shortage spell survival (logrank p = 0.046)"); plt.tight_layout(); plt.show()
km_res = json.loads((EXP / "survival/km.json").read_text()); km_res

In [ ]:
cox = json.loads((EXP / "survival/cox.json").read_text())
print(f"Cox concordance: {cox['concordance']:.3f} | n={cox['n']}, events={cox['events']}")
pd.DataFrame(cox["summary"]).T.astype(float).round(3)

## 4. Onset classification + H3 trade ablation (Weeks 6–7)

Form-level units (ingredient × dosage form), 4-fold rolling-origin temporal validation,
paired feature-set comparison on identical rows.

In [ ]:
for tag in ["domestic", "trade"]:
    r = json.loads((EXP / f"ablation/metrics_form_{tag}.json").read_text())
    print(f"{tag:9s} logit ROC {r['p_logit']['roc_auc']:.3f}  PR-AUC {r['p_logit']['pr_auc']:.4f}"
          f"  Brier {r['p_logit']['brier']:.4f}   (n={r['n']:,}, pos={r['pos']})")

In [ ]:
h3 = json.loads((EXP / "ablation/h3_tests.json").read_text())
l = h3["p_logit"]
print("H3 (logistic, paired):")
print(f"  AUC domestic {l['auc_domestic']:.4f} -> +trade {l['auc_trade']:.4f}"
      f"  (dAUC {l['delta_auc']:+.4f}, DeLong p = {l['delong_p']:.3f})")
print(f"  dBrier {l['brier_diff_trade_minus_dom']:+.4f} (naive DM p = {l['dm_p']:.2g}"
      " — cluster-robust check pending, week 8)")

## 5. Where this stands

* **RQ1:** ROC ≈ 0.78 out-of-sample at form level; precision inherently low (0.04% base rate) → decision-curve framing.
* **H3:** clean null on onset discrimination (ΔAUC +0.001, p = 0.88).
* **RQ2:** median spell 28.8 months; injectables and origin-concentrated APIs persist longest (hormones HR 0.40, p = 0.08).
* **Emerging thesis:** *tariff exposure shapes how long shortages last, not which products go short next.*
* **Next:** event study around IEEPA (Feb/Mar + Nov 2025) and §232 (Apr 2026); cluster-robust DM; calibration; decision curves.

Weekly reports with full details: `reports/week{3,5,6,7}_*.md`, `reports/coverage_report.md`, `reports/verification_week2.md`.